# Render “Inference Optimization” at 2K in Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lroe/llm-inference/blob/main/notebooks/render_2k_colab.ipynb)

This notebook:

1. Mounts **Google Drive** first and asks where to save.
2. Installs Node.js + the Chromium libraries Remotion needs.
3. Clones the public [lroe/llm-inference](https://github.com/lroe/llm-inference) repo.
4. Renders the 18:48 film at **2560×1440 (2K)** in six muted chunks.
5. Saves every finished chunk to Google Drive immediately, so a disconnect doesn’t lose everything.
6. Joins the chunks, adds the original narration audio, and saves the final MP4 to Drive.

**Runtime:** CPU is fine. Expect several hours for 2K. Use **Runtime → Change runtime type → High-RAM** if available. Keep the tab open; re-running the render cell resumes from saved chunks.

---
## Step 1 — Connect Google Drive

Run the next cell. Colab will ask for Google Drive access; approve it.


In [ ]:
from google.colab import drive
from pathlib import Path

print("Mounting Google Drive — approve the access prompt if it appears...")
drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = Path('/content/drive/MyDrive')
default_name = 'InferenceOptimization'
folder_name = input(f'Drive folder name [{default_name}]: ').strip() or default_name

OUTPUT_DIR = DRIVE_ROOT / folder_name
CHUNK_DIR = OUTPUT_DIR / '2k_chunks'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

print(f'\nDrive output folder: {OUTPUT_DIR}')
print(f'Chunk folder:        {CHUNK_DIR}')


---
## Step 2 — Install the rendering runtime

This installs Node.js 20, ffmpeg, and the system libraries Chromium needs. It takes a few minutes on a fresh Colab runtime.


In [ ]:
%%bash
set -euo pipefail

export DEBIAN_FRONTEND=noninteractive
apt-get update -qq

DEPS="curl gnupg ca-certificates git ffmpeg libnss3 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libatspi2.0-0 libcups2 libdrm2 libgbm1 libasound2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libpango-1.0-0 libcairo2 libglib2.0-0 libx11-xcb1 libxshmfence1 libxext6 libxi6 libxtst6 libxrender1"
apt-get install -y -qq $DEPS || {
  echo "libasound2 not available; retrying with libasound2t64 (Ubuntu 24.04)"
  apt-get install -y -qq ${DEPS/libasound2/libasound2t64}
}

if ! command -v node >/dev/null 2>&1 || [ "$(node -v | cut -d. -f1 | tr -d v)" -lt 18 ]; then
  echo "Installing Node.js 20..."
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
  apt-get install -y -qq nodejs
fi

echo
node -v
npm -v
ffmpeg -version | head -1


---
## Step 3 — Clone the project and install packages

The repository includes `audio.wav`, the Remotion source, and the local fonts. `npm ci` uses the committed lockfile.


In [ ]:
%%bash
set -euo pipefail

REPO_DIR=/content/llm-inference
REPO_URL=https://github.com/lroe/llm-inference.git

if [ -d "$REPO_DIR/.git" ]; then
  git -C "$REPO_DIR" fetch --depth 1 origin main
  git -C "$REPO_DIR" reset --hard origin/main
else
  git clone --depth 1 "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
npm ci --no-audit --no-fund

echo "Ensuring Remotion's headless browser is present..."
npx remotion browser ensure

echo
echo "Compositions:"
npx remotion compositions src/index.ts


---
## Step 4 — Render 2K chunks (resumable)

- Output is **2560×1440** (1920×1080 at scale 4/3).
- Audio is muted in the chunks; the original narration is muxed in the final step.
- Each finished chunk is copied to Drive immediately.
- Re-run this cell after a disconnect: chunks already on Drive are skipped.

> 2K rendering is CPU-heavy. Six chunks of ~5,640 frames; expect several hours total. Keep the Colab tab active.


In [ ]:
import json, os, shutil, subprocess, time
from pathlib import Path

REPO_DIR = Path('/content/llm-inference')
LOCAL_RENDER = Path('/content/render')
LOCAL_RENDER.mkdir(parents=True, exist_ok=True)

COMPOSITION = 'InferenceOptimization'
TOTAL_FRAMES = 33842
CHUNK_FRAMES = 5640
SCALE = 4 / 3                  # 1920x1080 -> 2560x1440 exactly
CRF = 17                       # lower is higher quality
FORCE_RERENDER = False         # set True to redo chunks already on Drive
CONCURRENCY = max(1, min(4, os.cpu_count() or 1))

chunks = []
start = 0
index = 0
while start < TOTAL_FRAMES:
    end = min(start + CHUNK_FRAMES - 1, TOTAL_FRAMES - 1)
    chunks.append((index, start, end))
    start = end + 1
    index += 1

print(f'Rendering {len(chunks)} chunks for {TOTAL_FRAMES} frames')
print(f'Resolution: 2560x1440 | CRF {CRF} | concurrency {CONCURRENCY}')
print(f'Output: {OUTPUT_DIR}')

manifest_path = OUTPUT_DIR / 'render_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
else:
    manifest = {'totalFrames': TOTAL_FRAMES, 'scale': 4/3, 'crf': CRF, 'chunks': {}}

for idx, start_frame, end_frame in chunks:
    drive_chunk = CHUNK_DIR / f'chunk_{idx:02d}_{start_frame}-{end_frame}.mp4'

    if (not FORCE_RERENDER) and drive_chunk.exists() and drive_chunk.stat().st_size > 1_000_000:
        print(f'[{idx+1}/{len(chunks)}] Skipping existing {drive_chunk.name}')
        manifest['chunks'][str(idx)] = {
            'start': start_frame, 'end': end_frame,
            'file': str(drive_chunk), 'status': 'complete'
        }
        manifest_path.write_text(json.dumps(manifest, indent=2))
        continue

    local_chunk = LOCAL_RENDER / f'chunk_{idx:02d}.mp4'
    local_log = LOCAL_RENDER / f'chunk_{idx:02d}.log'
    if local_chunk.exists():
        local_chunk.unlink()

    cmd = [
        'npx', 'remotion', 'render', 'src/index.ts', COMPOSITION, str(local_chunk),
        f'--frames={start_frame}-{end_frame}',
        f'--scale={SCALE}',
        '--codec=h264',
        f'--crf={CRF}',
        '--pixel-format=yuv420p',
        '--muted',
        '--overwrite',
        '--gl=swangle',
        f'--concurrency={CONCURRENCY}',
        '--image-format=jpeg',
        '--jpeg-quality=95',
        '--timeout=120000',
        '--log=info',
    ]

    print(f'\n[{idx+1}/{len(chunks)}] Rendering frames {start_frame}-{end_frame} ...')
    started = time.time()

    try:
        with local_log.open('w') as log_file:
            subprocess.run(cmd, cwd=REPO_DIR, stdout=log_file, stderr=subprocess.STDOUT, check=True)
    except subprocess.CalledProcessError:
        print(f'Chunk {idx} failed. Last log lines:')
        print('\n'.join(local_log.read_text(errors='replace').splitlines()[-30:]))
        raise

    if not local_chunk.exists() or local_chunk.stat().st_size < 1_000_000:
        raise RuntimeError(f'Chunk {idx} did not produce a valid MP4: {local_chunk}')

    elapsed = time.time() - started
    print(f'[{idx+1}/{len(chunks)}] Done in {elapsed/60:.1f} min. Saving to Drive...')
    shutil.copy2(local_chunk, drive_chunk)

    manifest['chunks'][str(idx)] = {
        'start': start_frame, 'end': end_frame,
        'file': str(drive_chunk), 'seconds': round(elapsed, 1), 'status': 'complete'
    }
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(f'[{idx+1}/{len(chunks)}] Saved {drive_chunk.name}')

print('\nAll chunks are on Google Drive.')


---
## Step 5 — Join chunks, add narration, save the final 2K MP4

This reads the six chunks from Drive, concatenates them without re-encoding the video, muxes `audio.wav`, and writes:

`<your Drive folder>/inference_optimization_2k.mp4`


In [ ]:
import json, shutil, subprocess
from pathlib import Path

REPO_DIR = Path('/content/llm-inference')
LOCAL_RENDER = Path('/content/render')
FINAL_LOCAL = LOCAL_RENDER / 'inference_optimization_2k.mp4'
FINAL_DRIVE = OUTPUT_DIR / 'inference_optimization_2k.mp4'
CONCAT_LIST = LOCAL_RENDER / 'concat.txt'

chunk_files = sorted(CHUNK_DIR.glob('chunk_*.mp4'))
expected_chunks = 6

print(f'Found {len(chunk_files)} chunk(s) in {CHUNK_DIR}')
if len(chunk_files) != expected_chunks:
    missing = expected_chunks - len(chunk_files)
    raise RuntimeError(f'Need {expected_chunks} chunks, found {len(chunk_files)}. Run the render cell again to finish the missing {missing}.')

CONCAT_LIST.write_text(''.join(f"file '{p}'\n" for p in chunk_files))
print('Joining video chunks...')
subprocess.run([
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(CONCAT_LIST),
    '-i', str(REPO_DIR / 'audio.wav'),
    '-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k',
    '-movflags', '+faststart', '-shortest', str(FINAL_LOCAL)
], check=True)

print('Copying final MP4 to Google Drive...')
shutil.copy2(FINAL_LOCAL, FINAL_DRIVE)

probe = subprocess.run([
    'ffprobe', '-v', 'error', '-select_streams', 'v:0',
    '-show_entries', 'stream=width,height,r_frame_rate',
    '-show_entries', 'format=duration,size',
    '-of', 'json', str(FINAL_DRIVE)
], capture_output=True, text=True, check=True)

info = json.loads(probe.stdout)
stream = info['streams'][0]
fmt = info['format']
print('\nFinal render:')
print(f'  Path:       {FINAL_DRIVE}')
print(f'  Resolution: {stream["width"]}x{stream["height"]}')
print(f'  Frame rate: {stream["r_frame_rate"]}')
print(f'  Duration:   {float(fmt["duration"])/60:.2f} min')
print(f'  Size:       {int(fmt["size"])/1024/1024:.1f} MB')

manifest_path = OUTPUT_DIR / 'render_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
else:
    manifest = {}
manifest['final'] = str(FINAL_DRIVE)
manifest['finalResolution'] = f'{stream["width"]}x{stream["height"]}'
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f'  Manifest:   {manifest_path}')


---
## Troubleshooting / resume

- **Colab disconnected:** reconnect, then re-run **Step 1** (Drive), **Step 2** (runtime), **Step 3** (clone), and **Step 4** (render). Finished chunks are skipped automatically.
- **A chunk is corrupt:** delete that `chunk_*.mp4` from `<Drive folder>/2k_chunks/`, then re-run Step 4.
- **Chrome/browser error:** re-run Step 2, then `npx remotion browser ensure` from `/content/llm-inference`.
- **Want to force a re-render:** set `FORCE_RERENDER = True` in Step 4.
- The final video is H.264 + AAC at 2560×1440, 30 fps. Chunks are muted; audio is added only in the final file.
